In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from rasterio.transform import from_origin
from rasterio import features
from rasterio.features import rasterize
import rasterio
import os

# sys.path.insert(0, "/ouce-home/projects/mistral/jamaica-ccri/processed_data/nbs-river-catchment/Connectivity")
import Robynlibrary as Robyn

# import sys
# sys.path.insert(0, "/ouce-home/projects/mistral/jamaica-ccri/processed_data/nbs-river-catchment/Connectivity")

import connectivity  # <-- from connectivity.py


In [ ]:
# base_path = Path("/ouce-home/projects/mistral/jamaica-ccri/processed_data/nbs-river-catchment/Connectivity")
base_path = Path()
connectivity_dir = base_path / "Connectivity_inputs"
inputs_dir = base_path / "other_inputs"
inputs_dir.mkdir(parents=True, exist_ok=True)
connectivity_results_dir = (base_path / "Connectivity" / "Results")
connectivity_results_dir.mkdir(parents=True, exist_ok=True)

connectivity_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
land_use_path = inputs_dir / "2013_landuse_Landcover.shp"
terrestrial_landcover = gpd.read_file(land_use_path).copy() # Optional: if you want to preserve the original
terrestrial_landcover.crs

In [ ]:
terrestrial_landcover.head()

In [ ]:
catchments_unionized_final = inputs_dir / "major_basins_plus_coastal_unionized_final.gpkg"
catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
print("Catchments:", len(catchments))
catchments.crs

# Condition factors

#### Normalized condition information 

In [ ]:
normalized_ones  = pd.read_csv(connectivity_dir / "landuse_condition_factors__normalized_ones_p4.csv")
normalized_zeros = pd.read_csv(connectivity_dir / "landuse_condition_factors__normalized_zeros_p4.csv")
normalized_ones.head()


#### Baseline condition info

In [ ]:
baseline_condition = pd.read_csv(connectivity_dir / "landuse_condition_factors__baseline_p4.csv")
baseline_condition.head()

In [ ]:
partial_reforest_condition = pd.read_csv(connectivity_dir / "landuse_condition_factors__partial_no_plantations_bamboo_p4.csv")
reforest_all_condition = pd.read_csv(connectivity_dir / "landuse_condition_factors__reforest_all_p4.csv")
long_timeframe_condition = pd.read_csv(connectivity_dir / "landuse_condition_factors__long_timeframe_p4.csv")


# Merge condition with land use

In [ ]:
landcover_normalized_ones = terrestrial_landcover.merge(normalized_ones, on='Classify')
landcover_normalized_zeros = terrestrial_landcover.merge(normalized_zeros, on='Classify')
landcover_baseline = terrestrial_landcover.merge(baseline_condition, on='Classify')
landcover_partial_reforest = terrestrial_landcover.merge(partial_reforest_condition, on='Classify')
landcover_reforest_all = terrestrial_landcover.merge(reforest_all_condition, on='Classify')
landcover_long_timeframe = terrestrial_landcover.merge(long_timeframe_condition, on='Classify')


# Convert land use condition files to rasters

In [ ]:
ones_raster = connectivity_dir / "landcover_normalized_ones.tif"
zeros_raster = connectivity_dir / "landcover_normalized_zeros.tif"
baseline_raster = connectivity_dir / "landcover_baseline.tif"
partial_reforest_raster = connectivity_dir / "landcover_partial_reforest.tif"
reforest_all_raster = connectivity_dir / "landcover_reforest_all.tif"
long_timeframe_raster = connectivity_dir / "landcover_long_timeframe.tif"

In [ ]:
# --- Check which column holds the condition values (no changes by default) ---
import pandas as pd
from pandas.api.types import is_numeric_dtype

def inspect_condition_col(gdf, name, prefer=None):
    print(f"\n[{name}] rows={len(gdf):,}")
    print("Columns:", list(gdf.columns))
    geom_col = gdf.geometry.name if hasattr(gdf, "geometry") else None

    # exact 'Condition'?
    if "Condition" in gdf.columns:
        print("Found exact column: 'Condition'")
        print(gdf[["Condition"]].head())
        return "Condition"

    # case-insensitive match
    ci = {c.lower(): c for c in gdf.columns}
    if "condition" in ci:
        print(f"Found case-insensitive match: '{ci['condition']}'")
        return ci["condition"]

    # user preference (e.g., 'Classify')
    if prefer and prefer in gdf.columns:
        print(f"Using preferred column: '{prefer}'")
        return prefer

    # otherwise: list numeric, non-geometry candidates
    numeric_cands = [c for c in gdf.columns
                     if c != geom_col and is_numeric_dtype(gdf[c])]
    print("Numeric candidates (non-geometry):", numeric_cands[:10])
    if numeric_cands:
        print(f"→ Likely choice: '{numeric_cands[0]}'")
        return numeric_cands[0]

    raise KeyError("No suitable condition/value column found.")

# Inspect the three GeoDataFrames you rasterize
col_ones  = inspect_condition_col(landcover_normalized_ones,  "normalized_ones",  prefer="Classify")
col_zeros = inspect_condition_col(landcover_normalized_zeros, "normalized_zeros", prefer="Classify")
col_base  = inspect_condition_col(landcover_baseline,         "baseline",         prefer="Classify")

print("\nChosen columns:", {"ones": col_ones, "zeros": col_zeros, "baseline": col_base})

# --- OPTIONAL: if Robyn.rasterize_condition requires a literal 'Condition', rename once ---
# landcover_normalized_ones  = landcover_normalized_ones.rename(columns={col_ones: "Condition"})
# landcover_normalized_zeros = landcover_normalized_zeros.rename(columns={col_zeros: "Condition"})
# landcover_baseline         = landcover_baseline.rename(columns={col_base: "Condition"})

In [ ]:
normalized_ones_condition_raster, transform = Robyn.rasterize_condition(
    landcover_normalized_ones,
    ones_raster
)

normalized_zeros_condition_raster, transform = Robyn.rasterize_condition(
    landcover_normalized_zeros,
    zeros_raster
)

baseline_condition_raster, transform = Robyn.rasterize_condition(
    landcover_baseline,
    baseline_raster
)

partial_reforest_condition_raster, transform = Robyn.rasterize_condition(
    landcover_partial_reforest,
    partial_reforest_raster
)

reforest_all_raster, transform = Robyn.rasterize_condition(
    landcover_reforest_all,
    reforest_all_raster
)


long_timeframe_raster, transform = Robyn.rasterize_condition(
    landcover_long_timeframe,
    long_timeframe_raster
)


# Resample to 100x100 size cells rather than current 10x10

In [ ]:
# Compute common spatial extent from baseline dataframe and output file names

bounds = landcover_normalized_ones.total_bounds

normalized_ones_resampled = connectivity_dir / "landcover_condition_normalized_ones_resampled.tif"

normalized_zeros_resampled = connectivity_dir / "landcover_condition_normalized_zeros_resampled.tif"

baseline_resampled = connectivity_dir / "landcover_condition_baseline_resampled.tif"

partial_reforest_resampled = connectivity_dir / "landcover_condition_partial_reforest_resampled.tif"

reforest_all_resampled = connectivity_dir / "landcover_condition_reforest_all_resampled.tif"

long_timeframe_resampled = connectivity_dir / "landcover_condition_long_timeframe_resampled.tif"


In [ ]:
# Do the resample and save

Robyn.resample_and_save(
    normalized_ones_condition_raster,
    transform,
    landcover_normalized_ones.crs,
    bounds,
    normalized_ones_resampled
)

Robyn.resample_and_save(
    normalized_zeros_condition_raster,
    transform,
    landcover_normalized_zeros.crs,
    bounds,
    normalized_zeros_resampled
)


Robyn.resample_and_save(
    baseline_condition_raster,
    transform,
    landcover_baseline.crs,
    bounds,
    baseline_resampled
)

Robyn.resample_and_save(
    partial_reforest_condition_raster,
    transform,
    landcover_partial_reforest.crs,
    bounds,
    partial_reforest_resampled
)

Robyn.resample_and_save(
    reforest_all_raster,
    transform,
    landcover_reforest_all.crs,
    bounds,
    reforest_all_resampled
)

Robyn.resample_and_save(
    long_timeframe_raster,
    transform,
    landcover_long_timeframe.crs,
    bounds,
    long_timeframe_resampled
)

# Connectivity analysis

In [ ]:
normalized_ones_connectivity = Robyn.compute_connectivity(normalized_ones_resampled)
normalized_zeros_connectivity = Robyn.compute_connectivity(normalized_zeros_resampled)
baseline_connectivity = Robyn.compute_connectivity(baseline_resampled)
partial_reforest_connectivity = Robyn.compute_connectivity(partial_reforest_resampled)
reforest_all_connectivity = Robyn.compute_connectivity(reforest_all_resampled)
long_timeframe_connectivity = Robyn.compute_connectivity(long_timeframe_resampled)

print(f"Ones connectivity: {normalized_ones_connectivity}")
print(f"Zeros connectivity: {normalized_zeros_connectivity}")
print(f"Baseline connectivity: {baseline_connectivity}")
print(f"Partial reforest connectivity: {partial_reforest_connectivity}")
print(f"Reforest all connectivity: {reforest_all_connectivity}")
print(f"Long timeframe connectivity: {long_timeframe_connectivity}")


In [ ]:
baseline_normalized_extremes = Robyn.calc_connectivity_normalized(
    baseline_connectivity,
    normalized_ones_connectivity,
    normalized_zeros_connectivity
)

partial_reforest_normalized_extremes = Robyn.calc_connectivity_normalized(
    partial_reforest_connectivity,
    normalized_ones_connectivity,
    normalized_zeros_connectivity
)

reforest_all_normalized_extremes = Robyn.calc_connectivity_normalized(
    reforest_all_connectivity,
    normalized_ones_connectivity,
    normalized_zeros_connectivity
)

long_timeframe_normalized_extremes = Robyn.calc_connectivity_normalized(
    long_timeframe_connectivity,
    normalized_ones_connectivity,
    normalized_zeros_connectivity
)


print(f"Baseline normalized (compared to extremes):         {baseline_normalized_extremes:.2f}%")
print(f"Partial reforest normalized (compared to extremes):         {partial_reforest_normalized_extremes:.2f}%")
print(f"Reforest all normalized (compared to extremes):         {reforest_all_normalized_extremes:.2f}%")
print(f"Long timeframes normalized (compared to extremes):         {long_timeframe_normalized_extremes:.2f}%")


# CATCHMENT LEVEL

In [ ]:
# Calculate the area in m² and km², and add them as new columns
catchments["area_m2"] = catchments.geometry.area
catchments["area_km2"] = catchments["area_m2"] / 1e6

# Add a new column with a unique new ID starting at 1
catchments["new_id"] = range(1, len(catchments) + 1)

# Calculate the equivalent diameter (distance across) in meters
# Equivalent diameter = 2 * sqrt(area_m2 / pi)
catchments["equiv_diam_m"] = 2 * np.sqrt(catchments["area_m2"] / np.pi)


# Calculate the smallest and largest HYBAS_ID area (in m² and km²)
smallest_area_m2 = catchments["area_m2"].min()
mean_area_m2 = catchments["area_m2"].mean()
largest_area_m2 = catchments["area_m2"].max()
smallest_area_km2 = catchments["area_km2"].min()
mean_area_km2 = catchments["area_km2"].mean()
largest_area_km2 = catchments["area_km2"].max()

smallest_diam = catchments["equiv_diam_m"].min()
mean_diam = catchments["equiv_diam_m"].mean()
largest_diam = catchments["equiv_diam_m"].max()


# Print summary statistics of hydrobasins

print("Smallest catchments area (m²):", smallest_area_m2)
print("Mean catchments area (m²):", mean_area_m2)
print("Largest catchments area (m²):", largest_area_m2)
print("Smallest catchments area (km²):", smallest_area_km2)
print("Mean catchments area (km²):", mean_area_km2)
print("Largest catchments area (km²):", largest_area_km2)

print("Smallest equivalent diameter (m):", smallest_diam)
print("Mean equivalent diameter (m):", mean_diam)
print("Largest equivalent diameter (m):", largest_diam)
number_catchments = catchments["new_id"].max()
print(number_catchments)


# # Write the modified hydrobasins to a new shapefile
# catchments.to_file("catchments_modified.shp")
catchments.to_file("catchments_modified.gpkg", driver="GPKG")


catchments.head()

In [ ]:
# Define the raster resolution (in meters) and extent
pixel_size = 100  # Change this value to your desired resolution (e.g., 10m)
minx, miny, maxx, maxy = catchments.total_bounds

# Compute width and height in terms of pixels
width = int(np.ceil((maxx - minx) / pixel_size))
height = int(np.ceil((maxy - miny) / pixel_size))

# Create an affine transform for the raster (origin at top-left)
transform = from_origin(minx, maxy, pixel_size, pixel_size)

# Create (geometry, value) pairs for rasterization using the "new_id" column as the value.
shapes = ((geom, value) for geom, value in zip(catchments.geometry, catchments['new_id']))

# Rasterize the geometries into a NumPy array.
raster = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,  # Pixels that don't fall within any geometry will be assigned the "fill" value (0).
    dtype=np.uint16  # Change type as needed based on your data range
)

In [ ]:
# Make sure these exist; they were created above
template_raster_candidates = [
    connectivity_dir / "landcover_condition_normalized_ones_resampled.tif",
    connectivity_dir / "landcover_condition_normalized_zeros_resampled.tif",
    connectivity_dir / "landcover_condition_baseline_resampled.tif",
    connectivity_dir / "landcover_condition_partial_reforest_resampled.tif",
    connectivity_dir / "landcover_condition_reforest_all_resampled.tif",
    connectivity_dir / "landcover_condition_long_timeframe_resampled.tif",
]

template_raster_path = next((p for p in template_raster_candidates if p.exists()), None)
if template_raster_path is None:
    any_tifs = sorted(connectivity_dir.glob("*.tif"))
    if not any_tifs:
        raise FileNotFoundError(f"No template raster found in {connectivity_dir}")
    template_raster_path = any_tifs[0]

print("Template raster:", template_raster_path)


In [ ]:
# --- Rasterize catchments to the template grid -------------------------------
catchments_raster_path = connectivity_dir / "catchments_on_template.tif"

with rasterio.open(template_raster_path) as tmpl:
    tmpl_crs       = tmpl.crs
    tmpl_transform = tmpl.transform
    tmpl_shape     = (tmpl.height, tmpl.width)
    tmpl_profile   = tmpl.profile

# Ensure catchments match CRS
gdf = catchments if catchments.crs == tmpl_crs else catchments.to_crs(tmpl_crs)

# Ensure stable contiguous integer ID named "catchment_uid"
if "catchment_uid" not in gdf.columns:
    gdf = gdf.reset_index(drop=True).copy()
    gdf["catchment_uid"] = np.arange(1, len(gdf) + 1, dtype=np.uint32)

# Burn catchment IDs to the template grid
burn = features.rasterize(
    ((geom, int(v)) for geom, v in zip(gdf.geometry, gdf["catchment_uid"])),
    out_shape=tmpl_shape,
    transform=tmpl_transform,
    fill=0,
    dtype="uint32",
    all_touched=False,
)

# Build a safe profile and write (try tiled first with valid block sizes)
width, height = tmpl_shape[1], tmpl_shape[0]
bx = min(256, max(16, (width  // 16) * 16 or 16))    # multiples of 16
by = min(256, max(16, (height // 16) * 16 or 16))

profile = tmpl_profile.copy()
profile.update(
    driver="GTiff",
    count=1,
    dtype=burn.dtype,
    nodata=0,
    compress="deflate",
    predictor=2,
    tiled=True,
    blockxsize=bx,
    blockysize=by,
)

try:
    with rasterio.open(catchments_raster_path, "w", **profile) as dst:
        dst.write(burn, 1)
except Exception:
    # Fallback: clean strip-based write (no tiling constraints)
    if catchments_raster_path.exists():
        try:
            catchments_raster_path.unlink()
        except Exception:
            pass
    clean = {
        "driver": "GTiff",
        "height": height,
        "width":  width,
        "count":  1,
        "dtype":  burn.dtype,
        "crs":    tmpl_crs,
        "transform": tmpl_transform,
        "nodata": 0,
        "compress": "deflate",
        "predictor": 2,
        "tiled": False,
    }
    with rasterio.open(catchments_raster_path, "w", **clean) as dst:
        dst.write(burn, 1)

print("Wrote catchments raster →", catchments_raster_path)

# --- Quick sanity printouts --------------------------------------------------
with rasterio.open(catchments_raster_path) as src, rasterio.open(template_raster_path) as tmpl:
    print("\nCatchments raster:")
    print("  CRS:", src.crs)
    print("  Size:", src.width, "×", src.height)
    print("  Pixel size:", src.transform.a, "×", src.transform.e)
    print("Template raster:")
    print("  CRS:", tmpl.crs)
    print("  Size:", tmpl.width, "×", tmpl.height)
    print("  Pixel size:", tmpl.transform.a, "×", tmpl.transform.e)

In [ ]:
# --- Clear names: paths vs arrays ---------------------------------
# These were already defined earlier as Paths to your resampled GeoTIFFs:
ones_path           = normalized_ones_resampled
zeros_path          = normalized_zeros_resampled
baseline_path       = baseline_resampled
partial_path        = partial_reforest_resampled
reforest_all_path   = reforest_all_resampled
long_timeframe_path = long_timeframe_resampled

# Collect paths
scenario_paths = {
    "ones":           ones_path,
    "zeros":          zeros_path,
    "baseline":       baseline_path,
    "partial":        partial_path,
    "reforest_all":   reforest_all_path,
    "long_timeframe": long_timeframe_path,
}

def grid_signature(p):
    with rasterio.open(p) as r:
        return (str(r.crs), r.transform, r.width, r.height)

grids = {name: grid_signature(path) for name, path in scenario_paths.items()}
baseline_grid = grids["baseline"]
mismatch = {name: g for name, g in grids.items() if g != baseline_grid}
print("Grid check:", "OK (all match)" if not mismatch else f"Mismatch in: {list(mismatch)}")

# Load arrays (readable, no name shadowing)
scenario_arrays = {
    name: Robyn.open_raster_as_array(str(path))
    for name, path in scenario_paths.items()
}

# Ensure catchment raster array is available
if "catchment_raster" not in globals():
    catchment_raster = Robyn.open_raster_as_array(str(catchments_raster_path))

In [ ]:
# Pick a few catchments to sanity-check normalization
test_ids = np.random.choice(np.unique(catchment_raster)[1:], size=min(3, len(np.unique(catchment_raster))-1), replace=False)
n_processes = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

for cid in test_ids:
    m = (catchment_raster == cid).astype(np.uint8)
    lam = 5  # or 0.5

    b = connectivity.landscape_connectivity(scenario_arrays["baseline"], n_processes, m, lam, "one_generation", 1)
    o = connectivity.landscape_connectivity(scenario_arrays["ones"],     n_processes, m, lam, "one_generation", 1)
    z = connectivity.landscape_connectivity(scenario_arrays["zeros"],    n_processes, m, lam, "one_generation", 1)
    n = Robyn.calc_connectivity_normalized(b, o, z)

    print(f"CID {int(cid)} | baseline={b:.3f}, ones={o:.3f}, zeros={z:.3f}, norm={n:.2f}%")

In [ ]:
# --- Per-catchment connectivity for ONE scenario (normalized to ones/zeros) --
# Requires:
#   - catchment_raster  (uint IDs on template grid, 0 = background)
#   - scenario_arrays = {"ones","zeros","baseline","partial","reforest_all","long_timeframe": arrays}
#   - catchments GeoDataFrame with area_m2, area_km2, equiv_diam_m

SCENARIO = "baseline"  # choose: "baseline", "partial", "reforest_all", "long_timeframe"
if SCENARIO not in set(scenario_arrays.keys()):
    raise ValueError(f"SCENARIO '{SCENARIO}' not found in scenario_arrays keys: {list(scenario_arrays.keys())}")

# Only require ones/zeros + this scenario
required_keys = {"ones", "zeros", SCENARIO}
missing = required_keys - set(scenario_arrays.keys())
if missing:
    raise KeyError(f"Missing rasters in scenario_arrays: {missing}")

# Processes (friendly to SLURM if present)
n_processes = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))
generations_mode = "one_generation"
number_of_species_generations = 1
LAMBDA = 50

# Catchment IDs (exclude 0)
catchment_ids = np.unique(catchment_raster)
catchment_ids = catchment_ids[catchment_ids > 0]
print("Running scenario:", SCENARIO, "| #catchments:", len(catchment_ids))

# Ensure attribute lookup
if "catchment_uid" not in catchments.columns:
    catchments = catchments.reset_index(drop=True).copy()
    catchments["catchment_uid"] = np.arange(1, len(catchments) + 1, dtype=np.uint32)
catch_attr = catchments.set_index("catchment_uid")[["area_m2","area_km2","equiv_diam_m"]]

rows = []

for cid in catchment_ids:
    mask = (catchment_raster == cid).astype(np.uint8)

    # Scenario connectivity for this catchment
    scenario_conn = connectivity.landscape_connectivity(
        scenario_arrays[SCENARIO], n_processes, mask, LAMBDA,
        generations_mode, number_of_species_generations
    )
    # Extremes for normalization
    ones_conn = connectivity.landscape_connectivity(
        scenario_arrays["ones"], n_processes, mask, LAMBDA,
        generations_mode, number_of_species_generations
    )
    zeros_conn = connectivity.landscape_connectivity(
        scenario_arrays["zeros"], n_processes, mask, LAMBDA,
        generations_mode, number_of_species_generations
    )

    # Normalize scenario (0–100%) against ones/zeros
    scenario_norm_pct = Robyn.calc_connectivity_normalized(scenario_conn, ones_conn, zeros_conn)

    # --- Print per-catchment results (both raw and % normalized) -------------
    print(f"CID {int(cid)} | connectivity={scenario_conn:.3f} | normalized={scenario_norm_pct:.2f}%", flush=True)

    a = catch_attr.loc[int(cid)]
    rows.append({
        "catchment_uid": int(cid),
        "area_m2": float(a["area_m2"]),
        "area_km2": float(a["area_km2"]),
        "equiv_diam_m": float(a["equiv_diam_m"]),
        "lambda": LAMBDA,
        "scenario": SCENARIO,
        "conn_scenario": scenario_conn,           # raw connectivity value
        "norm_scenario_pct": scenario_norm_pct,   # percentage (0–100)
    })

# Save CSVs
out = pd.DataFrame(rows).sort_values("catchment_uid").reset_index(drop=True)
# ▶ CHANGED: use connectivity_results_dir (was connectivity_dir)
out_path = connectivity_results_dir / f"catchment_connectivity__{SCENARIO}__lambda_{LAMBDA}.csv"
out.to_csv(out_path, index=False)
print(f"Saved:", out_path.resolve())

# ▶ CHANGED: use connectivity_results_dir (was connectivity_dir)
#all_df_path = connectivity_results_dir / f"catchment_connectivity__{SCENARIO}__all_lambdas.csv"
#out.to_csv(all_df_path, index=False)  # same content since only one lambda
#print(f"Saved combined:", all_df_path.resolve())